In [1]:
import os
import re
import glob
import itertools
import random
import pandas as pd
import numpy as np
import scipy
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

In [4]:
import sys        
sys.path.append('../tape_end2end/') 

from kinbert_dataset import BertDataset, BertLabeledDataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import math
from functools import reduce
import pickle 
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from tape import ProteinBertForSequenceClassification, TAPETokenizer
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt

from kinbert_mlm import MLM

from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
import seaborn as sn
import pandas as pd

from eval_utils import get_num_correct, createConfusionMatrix

In [5]:
print(torch.cuda.device_count())
print(torch.cuda.is_available())
print(torch.version.cuda)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
torch.cuda.empty_cache()

1
True
11.6


In [24]:
candidate_psites = pd.read_pickle("../../data/virus_dms/dms_candidate_psites.pkl")
candidate_psites.columns = ['p_ind', 'S', 'T', 'Y', 'entropy', 'neffective', 'wt_aa', 'substrate',
       'substrate_seq']
candidate_psites.to_pickle("../../data/virus_dms/dms_candidate_psites.pkl")

In [12]:
def eval_dataset_VSKinome(DATA_DIR,DATA_NAME_SUB,DATA_NAME_KIN,outdir,model_tag,dat_tag,psite_env = 7,entire_seq = False):
    dat_sub = pd.read_pickle(os.path.join(DATA_DIR,DATA_NAME_SUB))
    dat_kin = pd.read_pickle(os.path.join(DATA_DIR,DATA_NAME_KIN))
    dat_sub["p_ind"] = dat_sub["p_ind"].astype('int')
    kin_name_ls = []
    logit_array_ls = []
    for i in (tqdm(range(dat_kin.shape[0]))):
        s = dat_kin.iloc[[i],:]
        # print(s["kinase"].values)
        kin_name_ls.append(s["kinase"].values[0])
        dat_sub["kinase"] = s["kinase"].values[0]
        k_coord = s["kinase_coord"].values[0]
        # print(k_coord)
        dat_sub["kinase_seq"] = s["kinase_seq"].values[0]
        dat_sub["kinase_coord"] = s["kinase_coord"].values[0][0][0]+"-"+s["kinase_coord"].values[0][0][1]
        dat_sub["kinase_coord"] = dat_sub["kinase_coord"].apply(lambda s: [s.split("-")])
        if entire_seq:
            dat_sub["kinase_seq"] = s["kinase_seq"].values[0][int(k_coord[0][0])-1:int(k_coord[0][1])]
        dat = dat_sub
        dat["label"] = 1
        dataset = BertLabeledDataset(dat,psite_env = psite_env,check_psite = False,entire_seq = entire_seq)
        dataloader = DataLoader(dataset, 
                              batch_size=BATCH_SIZE, 
                              shuffle=False, 
                              collate_fn=dataset.collate_fn)
        running_vloss = 0.0
        total_vcorrect = 0
        n_vsample = 0
        model.train(False) # Don't need to track gradents for validation
        logits_preds = []
        y_preds = [] # save predction
        y_trues = [] # save ground truth
        class_probs = []
        class_labels = []
        for j, vdata in enumerate(dataloader, 0):
            batch_tokens, input_mask, targets = vdata['input_ids'], vdata['input_mask'], vdata['label']
            # print("data loaded")
            # print(batch_tokens.shape)
            (vloss, metrics),logits = model(input_ids = batch_tokens.to(device), input_mask=input_mask.to(device), targets=targets.to(device))
            # print("inference done")
            n_vsample += batch_tokens.shape[0]
            y_pred = (torch.max(torch.exp(logits), 1)[1]).data.cpu().numpy()
            y_preds.extend(y_pred)  # save prediction
            y_true = targets.data.cpu().numpy()
            y_trues.extend(y_true)
            logits_preds.append(logits.data.cpu().numpy())

        logits_preds = np.concatenate(logits_preds)
        logit_array_ls.append(logits_preds)
    with open(os.path.join(outdir,model_tag + "_" + dat_tag + "_logit"+ ".pkl"), 'wb') as f:
        pickle.dump(logit_array_ls, f)

    with open(os.path.join(outdir,model_tag + "_" + dat_tag + "_kinasename"+ ".pkl"), 'wb') as f:
        pickle.dump(kin_name_ls, f)
    
    print("finished!")
    print(os.path.join(outdir,model_tag + "_" + dat_tag + "_logit"+ ".pkl"))
    return 

## Inference using kinvertv2

In [11]:
# for vulcan
MODEL_DIR = "../../model/"

MODEL_TAG = "tapeClassify"

DATA_DIR = "../../data/virus_dms/"


OUT_DIR = "../../data/kinbertv2_pred"
model_tag = "kinbertv2"
BASE_MODEL = "tapeClassify_interations_e2e_1n1p_db_classed_random_lysate_random_30e_batchsize_8_lr_1e3_15aa"

dat_tags = ["virus_dms"]
sub_dat_paths = ["dms_candidate_psites.pkl"]
kin_dat_paths = ["../human_kinome_seq.pkl"]

psite_envs = [7]
entire_seq_boos = [False]
BATCH_SIZE = 1

In [8]:
model = ProteinBertForSequenceClassification.from_pretrained(MODEL_DIR+BASE_MODEL).to(device)
tokenizer = TAPETokenizer(vocab='iupac')

In [25]:
for i in range(len(dat_tags)):
    dat_tag = dat_tags[i]
    entire_seq_boo = entire_seq_boos[i]
    psite_env = psite_envs[i]
    DATA_NAME_SUB = sub_dat_paths[i]
    print(DATA_NAME_SUB)
    DATA_NAME_KIN = kin_dat_paths[i]
    eval_dataset_VSKinome(DATA_DIR,DATA_NAME_SUB,DATA_NAME_KIN,OUT_DIR,model_tag,dat_tag,psite_env = 7,entire_seq = entire_seq_boo)

dms_candidate_psites.pkl


100%|██████████| 472/472 [10:03<00:00,  1.28s/it]

finished!
../../data/kinbertv2_pred/kinbertv2_virus_dms_logit.pkl


In [28]:
human_kinome = pd.read_csv("../../data/human_kinome.csv")
uniprot2gene = {human_kinome["UniprotID"][i]:human_kinome["gene"][i] for i in range(human_kinome.shape[0])}
logits = pd.read_pickle("../../data/kinbertv2_pred/kinbertv2_virus_dms_logit.pkl")
kinase = pd.read_pickle("../../data/kinbertv2_pred/kinbertv2_virus_dms_kinasename.pkl")
substrate = pd.read_pickle("../../data/virus_dms/dms_candidate_psites.pkl")
assert len(logits[0]) == substrate.shape[0]
assert len(logits) == len(kinase)
m = torch.nn.Softmax()
pred_res = []
pred_res_prob = []
for i in range(substrate.shape[0]):
    s = substrate.iloc[i,:]
    predicted_kinase_prob = np.array([float(m(torch.tensor(logits_curr[i]))[1]) for logits_curr in logits])
    predicted_kinase_logits = np.array([logits_curr[i][1] for logits_curr in logits])
    kinase_sorted_ind = np.flipud(np.argsort(predicted_kinase_logits)) 
    kinase_sorted_ind[0:10]
    pred_logit = predicted_kinase_logits[kinase_sorted_ind[0:10]]
    pred_prob = predicted_kinase_prob[kinase_sorted_ind[0:10]]
    sel_ind = (pred_prob>0.85)
    pred_kinase = [uniprot2gene[kinase[i]] for i in kinase_sorted_ind[0:10]]
    pred_kinase_prob = [pred_kinase[i] + ":" + "{:.2f}".format(pred_logit[i])+":"+ "{:.2f}".format(pred_prob[i]) for i in range(len(pred_kinase)) if sel_ind[i]]
    pred_kinase_self = [pred_kinase[i]  for i in range(len(pred_kinase)) if sel_ind[i]]
    pred_res.append(pred_kinase)
    pred_res_prob.append(pred_kinase_prob)
substrate["pred_kinase"] = pred_res
substrate["pred_kinase_prob"] = pred_res_prob
substrate = substrate.loc[:,["substrate","p_ind","wt_aa","pred_kinase","pred_kinase_prob"]]
substrate["p_ind"] = substrate["p_ind"].astype('int')
substrate.to_csv("../../data/virus_dms/230105_dms_candidates_inference_kinbertv2.csv")

## Inference using kinbertV1

In [58]:
# for vulcan
MODEL_DIR = "../../model/"

MODEL_TAG = "tapeClassify"

DATA_DIR = "../../data/virus_dms/"


OUT_DIR = "../../data/kinbertv1_pred"
model_tag = "kinbertv1"
BASE_MODEL = "tapeClassify_interations_e2e_1n1p_classed_random_30e_batchsize_8_lr_1e3_15aa"

dat_tags = ["virus_dms"]
sub_dat_paths = ["dms_candidate_psites.pkl"]
kin_dat_paths = ["../human_kinome_seq.pkl"]

psite_envs = [7]
entire_seq_boos = [False]
BATCH_SIZE = 1

In [59]:
model = ProteinBertForSequenceClassification.from_pretrained(MODEL_DIR+BASE_MODEL).to(device)
tokenizer = TAPETokenizer(vocab='iupac')

In [60]:
for i in range(len(dat_tags)):
    dat_tag = dat_tags[i]
    entire_seq_boo = entire_seq_boos[i]
    psite_env = psite_envs[i]
    DATA_NAME_SUB = sub_dat_paths[i]
    print(DATA_NAME_SUB)
    DATA_NAME_KIN = kin_dat_paths[i]
    eval_dataset_VSKinome(DATA_DIR,DATA_NAME_SUB,DATA_NAME_KIN,OUT_DIR,model_tag,dat_tag,psite_env = 7,entire_seq = entire_seq_boo)

dms_candidate_psites.pkl


100%|██████████| 472/472 [10:03<00:00,  1.28s/it]

finished!
../../data/kinbertv1_pred/kinbertv1_virus_dms_logit.pkl


In [61]:
human_kinome = pd.read_csv("../../data/human_kinome.csv")
uniprot2gene = {human_kinome["UniprotID"][i]:human_kinome["gene"][i] for i in range(human_kinome.shape[0])}
logits = pd.read_pickle("../../data/kinbertv1_pred/kinbertv1_virus_dms_logit.pkl")
kinase = pd.read_pickle("../../data/kinbertv1_pred/kinbertv1_virus_dms_kinasename.pkl")
substrate = pd.read_pickle("../../data/virus_dms/dms_candidate_psites.pkl")
assert len(logits[0]) == substrate.shape[0]
assert len(logits) == len(kinase)
m = torch.nn.Softmax()
pred_res = []
pred_res_prob = []
for i in range(substrate.shape[0]):
    s = substrate.iloc[i,:]
    predicted_kinase_prob = np.array([float(m(torch.tensor(logits_curr[i]))[1]) for logits_curr in logits])
    predicted_kinase_logits = np.array([logits_curr[i][1] for logits_curr in logits])
    kinase_sorted_ind = np.flipud(np.argsort(predicted_kinase_logits)) 
    kinase_sorted_ind[0:10]
    pred_logit = predicted_kinase_logits[kinase_sorted_ind[0:10]]
    pred_prob = predicted_kinase_prob[kinase_sorted_ind[0:10]]
    sel_ind = (pred_prob>0.85)
    pred_kinase = [uniprot2gene[kinase[i]] for i in kinase_sorted_ind[0:10]]
    pred_kinase_prob = [pred_kinase[i] + ":" + "{:.2f}".format(pred_logit[i])+":"+ "{:.2f}".format(pred_prob[i]) for i in range(len(pred_kinase)) if sel_ind[i]]
    pred_kinase_self = [pred_kinase[i]  for i in range(len(pred_kinase)) if sel_ind[i]]
    pred_res.append(pred_kinase)
    pred_res_prob.append(pred_kinase_prob)
substrate["pred_kinase"] = pred_res
substrate["pred_kinase_prob"] = pred_res_prob
substrate = substrate.loc[:,["substrate","p_ind","wt_aa","pred_kinase","pred_kinase_prob"]]
substrate["p_ind"] = substrate["p_ind"].astype('int')
substrate.to_csv("../../data/virus_dms/230105_dms_candidates_inference_kinbertv1.csv")

<ipython-input-61-a12fc3d84b7d>:13: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  predicted_kinase_prob = np.array([float(m(torch.tensor(logits_curr[i]))[1]) for logits_curr in logits])
